<a href="https://colab.research.google.com/github/qayyumu/GradioExp/blob/main/Session-04/Gradio_and_Huggingface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup Environment

In [3]:
# %%capture
# !pip install gradio
# !pip install transformers


### Import necessary libraries 

In [1]:

import gradio as gr
from transformers import pipeline
import numpy as np
from dotenv import load_dotenv
import os
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")



/Users/mac14/data/code/GradioExp/Gradioexp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import os
from huggingface_hub import model_info, login
from dotenv import load_dotenv

load_dotenv()
my_token = os.getenv("HF_TOKEN")

# Force login for the session
login(token=my_token)

# Manually pass the token to model_info
info = model_info("meta-llama/Llama-3.2-3B", token=my_token)
total_bytes = sum(f.size for f in info.siblings if f.size is not None)

print(f"Total Size: {total_bytes / (1024**3):.2f} GB")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Total Size: 0.00 GB


### configure model pipeline to be used as backend

In [7]:
model = pipeline("text-generation")


def predict(prompt):
    completion = model(prompt)[0]["generated_text"]
    return completion

No model was supplied, defaulted to HuggingFaceTB/SmolLM3-3B and revision a07cc9a.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 326/326 [00:00<00:00, 5113.00it/s]


### Launch UI interface wiht desired pipeline

In [8]:
import gradio as gr

gr.Interface(fn=predict, inputs="text", outputs="text").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Audio to text Interface

transcriber = pipeline("automatic-speech-recognition", model="openai/whisper-base.en")


In [2]:
transcriber = pipeline("automatic-speech-recognition", model="openai/whisper-base.en")

Loading weights: 100%|██████████| 245/245 [00:00<00:00, 6168.80it/s]


In [3]:
def transcribe(audio):
    # try:
        sr, y = audio       
        # Convert to mono if stereo
        if y.ndim > 1:
            y = y.mean(axis=1)
            
        y = y.astype(np.float32)
        y /= np.max(np.abs(y))

        textReturn = transcriber({"sampling_rate": sr, "raw": y})["text"] 
        return textReturn  
    # except:
    #     return "problem detected"



In [4]:
demo = gr.Interface(
    transcribe,
    gr.Audio(sources="microphone"),
    "text",
    api_name="predict",
)


In [5]:
demo.launch(debug=True)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'tr

Keyboard interruption in main thread... closing server.


### Text To image 

In [7]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("image-text-to-text", model="Salesforce/blip-image-captioning-base")

Loading weights: 100%|██████████| 473/473 [00:00<00:00, 28144.50it/s]
The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when 

In [11]:
def predict(Imageinput, prompt):
    completion = pipe(Imageinput, prompt)
    return completion


In [ ]:
from PIL import Image
# import requests
# url = "https://raw.githubusercontent.com/timothybrooks/instruct-pix2pix/main/imgs/example.jpg"
# image = Image.open(requests.get(url, stream=True).raw)
img_path = "statue_of_liberty.jpg"
image = Image.open(img_path)
# image.show() # This will pop up the image on your computer
output = predict(image, prompt="what is in the image?")
print(output[0]['generated_text'])

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


what is in the image? - the statue of liberty


In [25]:
demo = gr.Interface(
    fn=predict, 
    inputs=[
        gr.Image(label="Upload Image", type="pil"), 
        gr.Textbox(label="Ask a question about the image", placeholder="What is in this image?")
    ], 
    outputs="text",
    title="Llama Vision Chat"
)
demo.launch(debug=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Keyboard interruption in main thread... closing server.
